In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Load Clean Inputs from Phase 2
clean_prices = pd.read_csv("clean_price_matrix(10).csv", parse_dates=["Date"], index_col="Date")
clean_returns = pd.read_csv("clean_return_matrix(10).csv", parse_dates=["Date"], index_col="Date")

tickers = clean_prices.columns.tolist()
num_assets = len(tickers)

# 2. Set Core Inputs
INITIAL_CAPITAL = 100_000.00  # $100,000 starting portfolio value
REBALANCE_FREQ = 'ME'          # Options: 'D' (Daily), 'W' (Weekly), 'ME' (Monthly), 'QE' (Quarterly), 'AE' (Annual/No rebalance)

In [2]:
PORTFOLIO_TYPE = "equal"  # Options: 'equal', 'custom', 'market_cap', 'long_short'

if PORTFOLIO_TYPE == "equal":
    # Option A: Equal-Weighted Portfolio
    weights = np.repeat(1.0 / num_assets, num_assets)

elif PORTFOLIO_TYPE == "custom":
    # Option B: User-Defined Weights (example: heavier tech allocation)
    custom_dict = {t: 1.0 for t in tickers}  # Default equal
    custom_dict['AAPL'], custom_dict['MSFT'], custom_dict['NVDA'] = 3.0, 3.0, 3.0
    
    raw_weights = np.array([custom_dict[t] for t in tickers])
    weights = raw_weights / raw_weights.sum()  # Normalize to sum to 1.0

elif PORTFOLIO_TYPE == "market_cap":
    # Option C: Market-Cap Weighted Portfolio
    mcaps = []
    for t in tickers:
        info = yf.Ticker(t).fast_info
        mcaps.append(info.market_cap if info.market_cap else 1.0)
    
    mcaps = np.array(mcaps)
    weights = mcaps / mcaps.sum()  # Normalize by market cap proportion

elif PORTFOLIO_TYPE == "long_short":
    # Option D: Long-Short Portfolio (Top 15 Long +0.0667, Bottom 15 Short -0.0667)
    # Uses trailing 1-year momentum to go long winners and short losers
    one_yr_returns = clean_prices.iloc[-1] / clean_prices.iloc[0] - 1
    ranked = one_yr_returns.rank()
    
    weights = np.where(ranked > (num_assets / 2), 1.0 / (num_assets / 2), -1.0 / (num_assets / 2))

weights_series = pd.Series(weights, index=tickers)

In [3]:
def simulate_portfolio(prices, initial_value, weights, freq='ME'):
    # Determine rebalance dates based on frequency
    if freq == 'D':
        rebalance_dates = prices.index
    else:
        rebalance_dates = prices.resample(freq).first().index

    portfolio_values = pd.Series(index=prices.index, dtype=float)
    current_cash = initial_value
    current_shares = pd.Series(0.0, index=prices.columns)

    for i, date in enumerate(prices.index):
        current_prices = prices.loc[date]

        # Rebalance on scheduled dates
        if date in rebalance_dates or i == 0:
            total_value = current_cash + (current_shares * current_prices).sum() if i > 0 else initial_value
            # Target dollar value per asset
            target_allocations = total_value * weights
            current_shares = target_allocations / current_prices
            current_cash = total_value - (current_shares * current_prices).sum()

        # Update daily total portfolio value
        portfolio_values.loc[date] = current_cash + (current_shares * current_prices).sum()

    return portfolio_values

daily_portfolio_value = simulate_portfolio(clean_prices, INITIAL_CAPITAL, weights_series, freq=REBALANCE_FREQ)
daily_portfolio_value.name = "Portfolio_Value"

In [4]:
# 1. Daily Portfolio P&L ($)
daily_pnl = daily_portfolio_value.diff()
daily_pnl.iloc[0] = daily_portfolio_value.iloc[0] - INITIAL_CAPITAL
daily_pnl.name = "Daily_PnL"

# 2. Daily Portfolio Simple Returns (%)
daily_returns = daily_portfolio_value.pct_change().fillna(0.0)
daily_returns.name = "Daily_Return"

# Combine into output DataFrame
portfolio_summary = pd.DataFrame({
    "Portfolio_Value": daily_portfolio_value,
    "Daily_PnL": daily_pnl,
    "Daily_Return": daily_returns
})

In [5]:
# Save Phase 3 outputs
portfolio_summary.to_csv("portfolio_phase3_output(10).csv")

print("--- Phase 3 Portfolio Summary ---")
print(f"Strategy: {PORTFOLIO_TYPE.upper()}")
print(f"Initial Capital: ${INITIAL_CAPITAL:,.2f}")
print(f"Final Value:     ${daily_portfolio_value.iloc[-1]:,.2f}")
print(f"Total P&L:       ${daily_portfolio_value.iloc[-1] - INITIAL_CAPITAL:,.2f}")
print("\nPreview:")
print(portfolio_summary.head())

--- Phase 3 Portfolio Summary ---
Strategy: EQUAL
Initial Capital: $100,000.00
Final Value:     $719,496.37
Total P&L:       $619,496.37

Preview:
            Portfolio_Value   Daily_PnL  Daily_Return
Date                                                 
2016-07-29    100000.000000    0.000000      0.000000
2016-08-01    100256.274997  256.274997      0.002563
2016-08-02     99747.931880 -508.343117     -0.005070
2016-08-03    100402.518171  654.586291      0.006562
2016-08-04    100434.964602   32.446431      0.000323
